# MT v3: Stronger Chinese → Vietnamese translation notebook

This notebook replaces the incomplete v2 pipeline with a stronger **from-scratch** setup that stays within the contest rules:

- **No pretrained translation models**
- Separate **SentencePiece** tokenizers trained on the provided train split
- A stronger **Transformer encoder-decoder**
- **Label smoothing**, **mixed precision**, **gradient clipping**, and **warmup + cosine decay**
- **Validation SacreBLEU** model selection
- **Top-k checkpoint averaging**
- **Beam search** with **length penalty** and **no-repeat n-gram blocking**
- Correct **`submission.csv`** + **`submission.zip`** export for the OJ

Recommended runtime: GPU (T4 / L4 / A100).  
If memory is tight, reduce `CFG.batch_size`, `CFG.d_model`, or `CFG.num_layers`.

In [53]:
import importlib
import subprocess
import sys

def ensure_package(module_name: str, pip_name: str | None = None):
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module_name])

ensure_package("sentencepiece")
ensure_package("sacrebleu")
ensure_package("pandas")
ensure_package("tqdm")

In [54]:
import gc
import json
import math
import os
import random
import re
import shutil
import unicodedata
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd
import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 67

def seed_everything(seed: int = SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()

print("device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

device: cuda
gpu: NVIDIA GeForce RTX 4050 Laptop GPU


## Configuration

In [55]:
@dataclass
class CFG:
    # Paths
    data_dir: str = "dataset"
    work_dir: str = "mt_artifacts"

    # Tokenization
    src_vocab_size: int = 4092
    tgt_vocab_size: int = 4092
    sp_model_type: str = "unigram"   # "unigram" or "bpe"
    character_coverage_zh: float = 0.9995

    # Length limits
    max_src_len: int = 160
    max_tgt_len: int = 180

    # Model
    d_model: int = 384
    nhead: int = 6
    num_encoder_layers: int = 5
    num_decoder_layers: int = 5
    dim_feedforward: int = 1536
    dropout: float = 0.15

    # Optimization
    epochs: int = 18
    batch_size: int = 80
    grad_accum_steps: int = 2
    lr: float = 3e-4
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.08
    label_smoothing: float = 0.1
    grad_clip: float = 1.0

    # Validation / decoding
    val_ratio: float = 0.05
    beam_size: int = 4
    length_penalty_alpha: float = 0.7
    no_repeat_ngram_size: int = 3
    max_decode_len: int = 180
    save_top_k: int = 3
    eval_every_epoch: bool = True

    # Practical knobs
    num_workers: int = 2
    pin_memory: bool = True
    train_shuffle: bool = True

cfg = CFG()
print(json.dumps(asdict(cfg), indent=2))

{
  "data_dir": "dataset",
  "work_dir": "mt_artifacts",
  "src_vocab_size": 4092,
  "tgt_vocab_size": 4092,
  "sp_model_type": "unigram",
  "character_coverage_zh": 0.9995,
  "max_src_len": 160,
  "max_tgt_len": 180,
  "d_model": 384,
  "nhead": 6,
  "num_encoder_layers": 5,
  "num_decoder_layers": 5,
  "dim_feedforward": 1536,
  "dropout": 0.15,
  "epochs": 18,
  "batch_size": 80,
  "grad_accum_steps": 2,
  "lr": 0.0003,
  "weight_decay": 0.0001,
  "warmup_ratio": 0.08,
  "label_smoothing": 0.1,
  "grad_clip": 1.0,
  "val_ratio": 0.05,
  "beam_size": 4,
  "length_penalty_alpha": 0.7,
  "no_repeat_ngram_size": 3,
  "max_decode_len": 180,
  "save_top_k": 3,
  "eval_every_epoch": true,
  "num_workers": 2,
  "pin_memory": true,
  "train_shuffle": true
}


## Data utilities

In [56]:
def find_dataset_dir(preferred: str = "dataset") -> Path:
    candidates = [
        Path(preferred),
        Path("/content/dataset"),
        Path("./dataset"),
        Path("../dataset"),
        Path("/kaggle/input/dataset/dataset"),
    ]
    for path in candidates:
        if (path / "train" / "train.zh").exists() and (path / "train" / "train.vi").exists():
            return path.resolve()
    raise FileNotFoundError(
        "Could not find dataset/. Expected train/train.zh, train/train.vi, and test/test.zh. "
        "Set CFG.data_dir to the correct folder."
    )

def read_lines(path: Path) -> list[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def write_lines(path: Path, lines: Iterable[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(f"{line}\n")

def normalize_zh(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = text.replace("\u3000", " ")
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = text.replace("…", "...")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = text.replace("\u3000", " ")
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = text.replace("…", "...")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r"\s+([)])", r"\1", text)
    text = re.sub(r"\s+([/])\s+", r"\1", text)
    return text.strip()

def postprocess_vi(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r"\s+([)])", r"\1", text)
    text = re.sub(r"\s+([/])\s+", r"\1", text)
    text = re.sub(r'\s+"', ' "', text)
    text = re.sub(r'"\s+', '" ', text)
    text = re.sub(r"\s+'", " '", text)
    text = re.sub(r"'\s+", "' ", text)
    return text.strip()

def should_copy_source(src: str) -> bool:
    s = src.strip()
    if not s:
        return True
    if re.search(r"(https?://|www\.|@)", s):
        return True
    # If there are no CJK characters, it is usually safer to copy verbatim.
    if not re.search(r"[\u3400-\u4dbf\u4e00-\u9fff]", s):
        return True
    return False

data_dir = find_dataset_dir(cfg.data_dir)
cfg.data_dir = str(data_dir)

train_zh_path = data_dir / "train" / "train.zh"
train_vi_path = data_dir / "train" / "train.vi"
test_zh_path = data_dir / "test" / "test.zh"

raw_train_zh = read_lines(train_zh_path)
raw_train_vi = read_lines(train_vi_path)
raw_test_zh = read_lines(test_zh_path)

assert len(raw_train_zh) == len(raw_train_vi), "train.zh and train.vi must have identical line counts"

train_zh = [normalize_zh(x) for x in raw_train_zh]
train_vi = [normalize_vi(x) for x in raw_train_vi]
test_zh = [normalize_zh(x) for x in raw_test_zh]

print(f"dataset dir : {data_dir}")
print(f"train pairs : {len(train_zh):,}")
print(f"test lines  : {len(test_zh):,}")
print("sample zh   :", train_zh[0] if train_zh else "(empty)")
print("sample vi   :", train_vi[0] if train_vi else "(empty)")

dataset dir : /home/izu/Projects/olpai/trans/dataset
train pairs : 25,648
test lines  : 6,413
sample zh   : 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ?
sample vi   : Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không?


In [57]:
def make_train_val_split(src_lines: list[str], tgt_lines: list[str], val_ratio: float, seed: int = SEED):
    assert len(src_lines) == len(tgt_lines)
    indices = list(range(len(src_lines)))
    rng = random.Random(seed)
    rng.shuffle(indices)

    val_size = max(1, int(len(indices) * val_ratio))
    val_idx = set(indices[:val_size])

    train_src, train_tgt, val_src, val_tgt = [], [], [], []
    for i, (s, t) in enumerate(zip(src_lines, tgt_lines)):
        if i in val_idx:
            val_src.append(s)
            val_tgt.append(t)
        else:
            train_src.append(s)
            train_tgt.append(t)

    return train_src, train_tgt, val_src, val_tgt

tr_src, tr_tgt, va_src, va_tgt = make_train_val_split(train_zh, train_vi, cfg.val_ratio, seed=SEED)

print(f"train split : {len(tr_src):,}")
print(f"valid split : {len(va_src):,}")

train split : 24,366
valid split : 1,282


## Train SentencePiece tokenizers from scratch

In [58]:
work_dir = Path(cfg.work_dir)
sp_dir = work_dir / "spm"
ckpt_dir = work_dir / "checkpoints"
pred_dir = work_dir / "predictions"

for p in [work_dir, sp_dir, ckpt_dir, pred_dir]:
    p.mkdir(parents=True, exist_ok=True)

sp_train_src = sp_dir / "train.norm.zh"
sp_train_tgt = sp_dir / "train.norm.vi"

write_lines(sp_train_src, tr_src)
write_lines(sp_train_tgt, tr_tgt)

src_prefix = sp_dir / "src_sp"
tgt_prefix = sp_dir / "tgt_sp"

if not (src_prefix.with_suffix(".model")).exists():
    spm.SentencePieceTrainer.train(
        input=str(sp_train_src),
        model_prefix=str(src_prefix),
        vocab_size=cfg.src_vocab_size,
        model_type=cfg.sp_model_type,
        character_coverage=cfg.character_coverage_zh,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
        split_digits=True,
        shuffle_input_sentence=True,
        input_sentence_size=min(len(tr_src), 1_000_000),
        train_extremely_large_corpus=False,
    )

if not (tgt_prefix.with_suffix(".model")).exists():
    spm.SentencePieceTrainer.train(
        input=str(sp_train_tgt),
        model_prefix=str(tgt_prefix),
        vocab_size=cfg.tgt_vocab_size,
        model_type=cfg.sp_model_type,
        character_coverage=1.0,
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
        pad_piece="<pad>",
        unk_piece="<unk>",
        bos_piece="<bos>",
        eos_piece="<eos>",
        split_digits=True,
        shuffle_input_sentence=True,
        input_sentence_size=min(len(tr_tgt), 1_000_000),
        train_extremely_large_corpus=False,
    )

sp_src = spm.SentencePieceProcessor(model_file=str(src_prefix.with_suffix(".model")))
sp_tgt = spm.SentencePieceProcessor(model_file=str(tgt_prefix.with_suffix(".model")))

PAD_ID = sp_src.pad_id()
UNK_ID = sp_src.unk_id()
BOS_ID = sp_src.bos_id()
EOS_ID = sp_src.eos_id()

assert PAD_ID == 0 and UNK_ID == 1 and BOS_ID == 2 and EOS_ID == 3

print("src vocab:", sp_src.vocab_size())
print("tgt vocab:", sp_tgt.vocab_size())

sample = tr_src[0]
print("src sample:", sample)
print("src ids   :", sp_src.encode(sample, out_type=int)[:30])
print("tgt sample:", tr_tgt[0])
print("tgt ids   :", sp_tgt.encode(tr_tgt[0], out_type=int)[:30])

src vocab: 4092
tgt vocab: 4092
src sample: 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ?
src ids   : [31, 241, 26, 1087, 1110, 8, 805, 688, 1015, 10, 7]
tgt sample: Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không?
tgt ids   : [611, 4, 236, 22, 10, 23, 25, 1298, 47, 4, 578, 1863, 667, 1586, 104, 4, 3084, 4, 3368, 59, 93, 1631, 7, 6]


## Dataset and dataloaders

In [59]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        src_lines: list[str],
        tgt_lines: list[str] | None,
        sp_src: spm.SentencePieceProcessor,
        sp_tgt: spm.SentencePieceProcessor | None,
        max_src_len: int,
        max_tgt_len: int,
    ):
        self.src_lines = src_lines
        self.tgt_lines = tgt_lines
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.src_lines)

    def encode_src(self, text: str) -> list[int]:
        ids = self.sp_src.encode(text, out_type=int)[: self.max_src_len - 2]
        return [BOS_ID] + ids + [EOS_ID]

    def encode_tgt(self, text: str) -> list[int]:
        assert self.sp_tgt is not None
        ids = self.sp_tgt.encode(text, out_type=int)[: self.max_tgt_len - 2]
        return [BOS_ID] + ids + [EOS_ID]

    def __getitem__(self, idx: int):
        src_ids = self.encode_src(self.src_lines[idx])
        item = {"src_ids": src_ids, "src_text": self.src_lines[idx]}
        if self.tgt_lines is not None:
            tgt_ids = self.encode_tgt(self.tgt_lines[idx])
            item["tgt_ids"] = tgt_ids
            item["tgt_text"] = self.tgt_lines[idx]
        return item

def pad_sequences(seqs: list[list[int]], pad_value: int = PAD_ID) -> torch.Tensor:
    max_len = max(len(x) for x in seqs)
    arr = torch.full((len(seqs), max_len), pad_value, dtype=torch.long)
    for i, seq in enumerate(seqs):
        arr[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
    return arr

def collate_fn(batch: list[dict]):
    # Sort by source length (descending) to reduce padding a bit.
    batch = sorted(batch, key=lambda x: len(x["src_ids"]), reverse=True)
    src = pad_sequences([x["src_ids"] for x in batch], PAD_ID)
    out = {
        "src": src,
        "src_text": [x["src_text"] for x in batch],
    }
    if "tgt_ids" in batch[0]:
        tgt = pad_sequences([x["tgt_ids"] for x in batch], PAD_ID)
        out["tgt"] = tgt
        out["tgt_text"] = [x["tgt_text"] for x in batch]
    return out

train_ds = TranslationDataset(tr_src, tr_tgt, sp_src, sp_tgt, cfg.max_src_len, cfg.max_tgt_len)
valid_ds = TranslationDataset(va_src, va_tgt, sp_src, sp_tgt, cfg.max_src_len, cfg.max_tgt_len)
test_ds = TranslationDataset(test_zh, None, sp_src, None, cfg.max_src_len, cfg.max_tgt_len)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=cfg.train_shuffle,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    collate_fn=collate_fn,
)

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))
print("test  batches:", len(test_loader))

train batches: 305
valid batches: 17
test  batches: 81


## Transformer model

In [60]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

class TransformerMT(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 384,
        nhead: int = 6,
        num_encoder_layers: int = 5,
        num_decoder_layers: int = 5,
        dim_feedforward: int = 1536,
        dropout: float = 0.15,
        pad_id: int = 0,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
        self.generator.weight = self.tgt_emb.weight
        self.dropout = nn.Dropout(dropout)

        self._reset_parameters()

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def src_padding_mask(self, src: torch.Tensor) -> torch.Tensor:
        return src.eq(self.pad_id)

    def tgt_padding_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        return tgt.eq(self.pad_id)

    def causal_mask(self, size: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.full((size, size), float("-inf"), device=device), diagonal=1)

    def encode(self, src: torch.Tensor):
        src_pad = self.src_padding_mask(src)
        src_x = self.dropout(self.pos(self.src_emb(src) * math.sqrt(self.d_model)))
        memory = self.transformer.encoder(src_x, src_key_padding_mask=src_pad)
        return memory, src_pad

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, src_pad: torch.Tensor):
        tgt_pad = self.tgt_padding_mask(tgt)
        tgt_mask = self.causal_mask(tgt.size(1), tgt.device)
        tgt_x = self.dropout(self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model)))
        out = self.transformer.decoder(
            tgt=tgt_x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )
        logits = self.generator(out)
        return logits

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor):
        memory, src_pad = self.encode(src)
        logits = self.decode(tgt_in, memory, src_pad)
        return logits

model = TransformerMT(
    src_vocab_size=sp_src.vocab_size(),
    tgt_vocab_size=sp_tgt.vocab_size(),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    dim_feedforward=cfg.dim_feedforward,
    dropout=cfg.dropout,
    pad_id=PAD_ID,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params:,}")

trainable params: 23,849,472


## Training helpers

In [61]:
def shift_tgt_for_teacher_forcing(tgt: torch.Tensor):
    return tgt[:, :-1], tgt[:, 1:]

def compute_loss(logits: torch.Tensor, tgt_out: torch.Tensor, label_smoothing: float = 0.1):
    vocab_size = logits.size(-1)
    return F.cross_entropy(
        logits.reshape(-1, vocab_size),
        tgt_out.reshape(-1),
        ignore_index=PAD_ID,
        label_smoothing=label_smoothing,
    )

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_optimizer_and_scheduler(model: nn.Module, cfg: CFG, total_steps: int):
    optimizer = AdamW(
        model.parameters(),
        lr=cfg.lr,
        betas=(0.9, 0.98),
        eps=1e-9,
        weight_decay=cfg.weight_decay,
    )

    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))

    def lr_lambda(current_step: int):
        step = current_step + 1
        if step <= warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler

def save_checkpoint(path: Path, model: nn.Module, optimizer, scheduler, epoch: int, bleu: float):
    state = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "bleu": bleu,
        "cfg": asdict(cfg),
    }
    torch.save(state, path)

def get_top_checkpoint_paths(ckpt_dir: Path) -> list[Path]:
    scored = []
    for p in ckpt_dir.glob("epoch*_bleu*.pt"):
        m = re.search(r"bleu([0-9]+(?:\.[0-9]+)?)", p.stem)
        if m:
            scored.append((float(m.group(1)), p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored]

def average_checkpoints(ckpt_paths: list[Path], out_path: Path):
    assert ckpt_paths, "No checkpoints to average"
    avg_state = None
    n = 0

    for path in ckpt_paths:
        state = torch.load(path, map_location="cpu")
        model_state = state["model_state"]
        if avg_state is None:
            avg_state = {k: v.clone().float() for k, v in model_state.items()}
        else:
            for k in avg_state:
                avg_state[k] += model_state[k].float()
        n += 1

    for k in avg_state:
        avg_state[k] /= n

    torch.save({"model_state": avg_state, "paths": [str(p) for p in ckpt_paths]}, out_path)
    return out_path

## Beam search + SacreBLEU evaluation

In [62]:
def _has_repeat_ngram(tokens: list[int], cand: int, n: int) -> bool:
    if n <= 0 or len(tokens) + 1 < n:
        return False
    trial = tokens + [cand]
    target_ngram = tuple(trial[-n:])
    for i in range(len(trial) - n):
        if tuple(trial[i : i + n]) == target_ngram:
            return True
    return False

@torch.no_grad()
def beam_search_decode(
    model: TransformerMT,
    src_text: str,
    sp_src: spm.SentencePieceProcessor,
    sp_tgt: spm.SentencePieceProcessor,
    beam_size: int = 4,
    max_len: int = 180,
    alpha: float = 0.7,
    no_repeat_ngram_size: int = 3,
) -> str:
    if should_copy_source(src_text):
        return src_text

    src_ids = [BOS_ID] + sp_src.encode(src_text, out_type=int)[: cfg.max_src_len - 2] + [EOS_ID]
    src = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    memory, src_pad = model.encode(src)

    beams = [([BOS_ID], 0.0, False)]
    finished = []

    def norm_score(score: float, length: int) -> float:
        length = max(length, 1)
        return score / (((5 + length) / 6) ** alpha)

    for _ in range(max_len):
        candidates = []
        for tokens, score, done in beams:
            if done:
                candidates.append((tokens, score, done))
                continue

            tgt = torch.tensor(tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = model.decode(tgt, memory, src_pad)[0, -1]
            log_probs = F.log_softmax(logits, dim=-1)
            log_probs[UNK_ID] = -1e9

            if len(tokens) < 2:
                log_probs[EOS_ID] = -1e9

            topk_scores, topk_ids = torch.topk(log_probs, k=beam_size * 2)
            for lp, token_id in zip(topk_scores.tolist(), topk_ids.tolist()):
                if no_repeat_ngram_size > 0 and _has_repeat_ngram(tokens, token_id, no_repeat_ngram_size):
                    continue
                new_tokens = tokens + [token_id]
                new_score = score + lp
                is_done = token_id == EOS_ID
                candidates.append((new_tokens, new_score, is_done))

        candidates.sort(key=lambda x: norm_score(x[1], len(x[0])), reverse=True)
        beams = candidates[:beam_size]

        finished.extend([b for b in beams if b[2]])
        if len(finished) >= beam_size:
            best_finished = max(finished, key=lambda x: norm_score(x[1], len(x[0])))
            best_active = max(beams, key=lambda x: norm_score(x[1], len(x[0])))
            if norm_score(best_finished[1], len(best_finished[0])) >= norm_score(best_active[1], len(best_active[0])):
                break

    pool = finished if finished else beams
    best_tokens, _, _ = max(pool, key=lambda x: norm_score(x[1], len(x[0])))

    if EOS_ID in best_tokens[1:]:
        best_tokens = best_tokens[1 : best_tokens.index(EOS_ID)]
    else:
        best_tokens = best_tokens[1:]

    text = sp_tgt.decode(best_tokens)
    return postprocess_vi(text)

@torch.no_grad()
def predict_texts(model: TransformerMT, src_texts: list[str], desc: str = "decode") -> list[str]:
    model.eval()
    preds = []
    for src_text in tqdm(src_texts, desc=desc):
        pred = beam_search_decode(
            model=model,
            src_text=src_text,
            sp_src=sp_src,
            sp_tgt=sp_tgt,
            beam_size=cfg.beam_size,
            max_len=cfg.max_decode_len,
            alpha=cfg.length_penalty_alpha,
            no_repeat_ngram_size=cfg.no_repeat_ngram_size,
        )
        preds.append(pred)
    return preds

def corpus_bleu_score(preds: list[str], refs: list[str]) -> float:
    return sacrebleu.corpus_bleu(preds, [refs]).score

## Training loop

In [63]:
def train_one_epoch(
    model: TransformerMT,
    loader: DataLoader,
    optimizer,
    scheduler,
    scaler,
    epoch_idx: int,
    cfg: CFG,
):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f"train epoch {epoch_idx}", leave=False)
    for step, batch in enumerate(pbar, start=1):
        src = batch["src"].to(DEVICE, non_blocking=True)
        tgt = batch["tgt"].to(DEVICE, non_blocking=True)
        tgt_in, tgt_out = shift_tgt_for_teacher_forcing(tgt)

        with autocast(enabled=AMP_ENABLED):
            logits = model(src, tgt_in)
            loss = compute_loss(logits, tgt_out, cfg.label_smoothing)
            loss = loss / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if step % cfg.grad_accum_steps == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * cfg.grad_accum_steps
        avg_loss = running_loss / step
        pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    return running_loss / max(1, len(loader))

@torch.no_grad()
def evaluate_bleu(model: TransformerMT, src_texts: list[str], ref_texts: list[str]) -> float:
    preds = predict_texts(model, src_texts, desc="valid decode")
    return corpus_bleu_score(preds, ref_texts)

def fit(model: TransformerMT, train_loader: DataLoader, valid_src: list[str], valid_tgt: list[str], cfg: CFG):
    total_updates = math.ceil(len(train_loader) / cfg.grad_accum_steps) * cfg.epochs
    optimizer, scheduler = build_optimizer_and_scheduler(model, cfg, total_updates)
    scaler = GradScaler(enabled=AMP_ENABLED)

    history = []
    best_bleu = -1.0
    saved_paths = []

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch, cfg)

        bleu = float("nan")
        if cfg.eval_every_epoch:
            bleu = evaluate_bleu(model, valid_src, valid_tgt)
            ckpt_name = f"epoch{epoch:02d}_bleu{bleu:.4f}.pt"
            ckpt_path = ckpt_dir / ckpt_name
            save_checkpoint(ckpt_path, model, optimizer, scheduler, epoch, bleu)
            saved_paths.append(ckpt_path)

            ranked = get_top_checkpoint_paths(ckpt_dir)
            for extra_path in ranked[cfg.save_top_k:]:
                if extra_path.exists():
                    extra_path.unlink()

            if bleu > best_bleu:
                best_bleu = bleu

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_bleu": bleu,
            "best_bleu": best_bleu,
        }
        history.append(record)
        print(record)

    return pd.DataFrame(history)

## Train, select top checkpoints, and average them

In [64]:
history_df = fit(model, train_loader, va_src, va_tgt, cfg)
history_df

/tmp/ipykernel_78560/2468231188.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP_ENABLED)
train epoch 1:   0%|          | 0/305 [00:00<?, ?it/s]/tmp/ipykernel_78560/2468231188.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):
valid decode: 100%|██████████| 1282/1282 [06:28<00:00,  3.30it/s]                         


{'epoch': 1, 'train_loss': 6.43121253466997, 'valid_bleu': 3.1445589203863356, 'best_bleu': 3.1445589203863356}


valid decode: 100%|██████████| 1282/1282 [05:45<00:00,  3.72it/s]                         


{'epoch': 2, 'train_loss': 4.713467332183337, 'valid_bleu': 7.248331041082146, 'best_bleu': 7.248331041082146}


valid decode: 100%|██████████| 1282/1282 [05:33<00:00,  3.84it/s]                         


...

valid decode: 100%|██████████| 1282/1282 [02:06<00:00, 10.16it/s]                          


{'epoch': 17, 'train_loss': 2.407019658166854, 'valid_bleu': 31.077533637476524, 'best_bleu': 31.077533637476524}


valid decode: 100%|██████████| 1282/1282 [02:06<00:00, 10.10it/s]                          


{'epoch': 18, 'train_loss': 2.3956261072002474, 'valid_bleu': 31.88851236680637, 'best_bleu': 31.88851236680637}


,epoch,train_loss,valid_bleu,best_bleu
0,1,6.431213,3.144559,3.144559
1,2,4.713467,7.248331,7.248331
2,3,4.067357,12.147427,12.147427
3,4,3.674146,15.750265,15.750265
4,5,3.396174,18.778738,18.778738
5,6,3.181808,21.435785,21.435785
6,7,3.015511,24.653611,24.653611
7,8,2.882955,26.274897,26.274897
8,9,2.772416,27.378169,27.378169
9,10,2.682449,28.386081,28.386081


In [65]:
top_ckpts = get_top_checkpoint_paths(ckpt_dir)[: cfg.save_top_k]
print("top checkpoints:")
for p in top_ckpts:
    print(" -", p.name)

avg_ckpt_path = ckpt_dir / "averaged_topk.pt"
average_checkpoints(top_ckpts, avg_ckpt_path)
print("averaged checkpoint saved to:", avg_ckpt_path)

top checkpoints:
 - epoch18_bleu31.8885.pt
 - epoch16_bleu31.4035.pt
 - epoch18_bleu31.3580.pt
averaged checkpoint saved to: mt_artifacts/checkpoints/averaged_topk.pt


## Load averaged checkpoint for final decoding

In [66]:
best_model = TransformerMT(
    src_vocab_size=sp_src.vocab_size(),
    tgt_vocab_size=sp_tgt.vocab_size(),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    dim_feedforward=cfg.dim_feedforward,
    dropout=cfg.dropout,
    pad_id=PAD_ID,
).to(DEVICE)

avg_state = torch.load(avg_ckpt_path, map_location=DEVICE)
best_model.load_state_dict(avg_state["model_state"], strict=True)
best_model.eval()

valid_preds = predict_texts(best_model, va_src, desc="valid preview decode")
valid_bleu = corpus_bleu_score(valid_preds, va_tgt)
print(f"averaged checkpoint valid BLEU: {valid_bleu:.4f}")

/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(
valid preview decode:   0%|          | 0/1282 [00:00<?, ?it/s]/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
valid preview decode: 100%|██████████| 1282/1282 [02:09<00:00,  9.90it/s]

averaged checkpoint valid BLEU: 8.9813


In [67]:
preview_df = pd.DataFrame(
    {
        "zh": va_src[:10],
        "reference_vi": va_tgt[:10],
        "pred_vi": valid_preds[:10],
    }
)
preview_df

,zh,reference_vi,pred_vi
0,我们 想 买 些 唱片 。,Chúng_tôi muốn mua một_số đĩa_hát.,chúng_tôi muốn mua mua một một_chút.
1,不 你 可以 到 药房 拿药 。,"Không, bạn có_thể mua nó ở nhà_thuốc.",không không được được được cho cho đến.
2,我 想 要 晚霜 。,Tôi muốn một_ít kem ban_đêm.,tôi muốn muốn một_ít kem.
3,今天 不 热 。,hôm_nay không nóng.,hôm_nay không nóng.
4,我 要 找 带有 Prada 金属 标签 的 东西 。,Tôi đang tìm thứ có nhãn kim_loại Prada.,Tôi đ đang tìm một một_sốsố màu màu đen...
5,不 是 很 贵 。,Nó không đắt lắm.,"Không, nó rất rất nhiều."
6,"好 的 , 我 会 去 下载 的 ,","Được, tôi sẽ tải.","Vâng,, tôi đi ra ra ra."
7,"请 妳挑 吧 , 我们 有 很多 款式 。","mời chị lựa đi, chúng_tôi có rất nhiều kiểu.","mời mời chị đi đi,, chúng_tôi rất rất rất nhiều."
8,我 想 知道 是否 有 今晚 七 点 靠 窗 的 桌子 。,Tôi tự hỏi liệu còn bàn gần cửa_sổ vào lúc 7 g...,Tôi muốn hỏi hỏi hỏi 7 giờ...
9,你 爸爸 作 什么 工作 ?,ba của bạn làm việc gì?,bạn bạn làm làm gì gì?


## Generate OJ submission

In [68]:
test_preds = predict_texts(best_model, test_zh, desc="test decode")

submission_df = pd.DataFrame(
    {
        "tieng_trung": raw_test_zh,  # keep original input lines in the CSV
        "tieng_viet": [postprocess_vi(x) for x in test_preds],
    }
)

assert len(submission_df) == len(raw_test_zh)
assert submission_df["tieng_viet"].isna().sum() == 0

submission_csv = pred_dir / "submission.csv"
submission_zip = pred_dir / "submission.zip"

submission_df.to_csv(submission_csv, index=False, encoding="utf-8")

with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_csv, arcname="submission.csv")

print("saved:", submission_csv.resolve())
print("saved:", submission_zip.resolve())
submission_df.head()

test decode:   0%|          | 0/6413 [00:00<?, ?it/s]/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
test decode: 100%|██████████| 6413/6413 [11:04<00:00,  9.65it/s]

saved: /home/izu/Projects/olpai/trans/mt_artifacts/predictions/submission.csv
saved: /home/izu/Projects/olpai/trans/mt_artifacts/predictions/submission.zip


,tieng_trung,tieng_viet
0,还 早 呀 ， 过 几 年 再 说 。,hay hay_là ra ra ra ngoài ngoài.
1,你 跟 我 去吧,bạn bạn đi ra ra
2,我 都 喜欢 。,tôi đ đều thích.
3,我 受伤 了 需要 一 辆 救护车 。,Tôi đã đã đã bị bị_thươngthươngthương..
4,等 一下 。 这个 是 吗 ？,đợi một_chút.


## Notes for squeezing a bit more BLEU

If you still have GPU time left, the safest next knobs to try are:

1. Increase `CFG.epochs` to **22-26**
2. Increase `CFG.beam_size` to **5**
3. Increase `CFG.d_model` to **512** (only if VRAM allows)
4. Switch `CFG.sp_model_type` between `"unigram"` and `"bpe"`
5. Keep checkpoint averaging enabled (`CFG.save_top_k = 3` or `5`)

For a strict OJ, the biggest practical wins usually come from:
- clean normalization
- a stable Transformer
- checkpoint averaging
- beam search with repetition blocking
- correct submission formatting